In [80]:
import pandas as pd
from sksurv.datasets import load_gbsg2
import numpy as np
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter, WeibullAFTFitter
from sksurv.util import Surv
from sklearn.model_selection import train_test_split
from class_DecisionTreeBaggingClassifier import DecisionTreeBaggingClassifier
import os, json
import matplotlib.pyplot as plt
from utils import ipc_weighted_mse

def create_data_with_ipc_weights(tau: float, data: pd.DataFrame) -> pd.DataFrame:
    # Fit the Kaplan-Meier estimator
    kmf = KaplanMeierFitter()
    kmf.fit(np.array(data["time"],dtype=float), event_observed=  np.array(1 - data["event"],dtype=bool))

    # Efficiently calculate the 'survived' column using np.select for vectorized operations
    conditions = [
        (data["time"] <= tau) & (data["event"] == 1),
        (data["time"] >= tau),
        (data["time"] <= tau) & (data["event"] == 0),
    ]
    choices = [0, 1, 999]
    data["survived"] = np.select(conditions, choices, default=999)

    # Calculate the IPCW weights
    survival_times = data["time"]
    survival_probabilities = kmf.survival_function_at_times(
        survival_times
    ).values.flatten()
    ipcw_weights = 1 / survival_probabilities
    ipcw_weight_tau = 1 / kmf.survival_function_at_times(tau).values.flatten()[0]

    # Apply weights based on the 'survived' column
    data["weights_ipcw"] = np.where(
        data["survived"] == 1,
        ipcw_weight_tau,
        np.where(data["survived"] == 0, ipcw_weights, 0),
    )

    # Normalize the weights
    data["weights_ipcw"] /= data["weights_ipcw"].sum()

    return data

In [92]:
tau = 745

# load dataset
data = load_gbsg2()
X, y = data
data_df = X.copy()

# Extract time and event from structured array
data_df['time'] = y['time'].astype(float)

# Note: after the swap in this notebook, y['cens'] == True means event observed
data_df['event'] = y['cens'].astype(bool)

df_train = create_data_with_ipc_weights(tau= tau, data = data_df)
print(df_train.survived.value_counts()/len(df_train))
df_train

survived
1      0.651603
0      0.246356
999    0.102041
Name: count, dtype: float64


,age,estrec,horTh,menostat,pnodes,progrec,tgrade,tsize,time,event,survived,weights_ipcw
0,70.0,66.0,no,Post,3.0,48.0,II,21.0,1814.0,True,1,0.001655
1,56.0,77.0,yes,Post,7.0,61.0,II,12.0,2018.0,True,1,0.001655
2,58.0,271.0,yes,Post,9.0,52.0,II,35.0,712.0,True,0,0.001615
3,59.0,29.0,yes,Post,4.0,60.0,II,17.0,1807.0,True,1,0.001655
4,73.0,65.0,no,Post,1.0,26.0,II,35.0,772.0,True,1,0.001655
...,...,...,...,...,...,...,...,...,...,...,...,...
681,49.0,84.0,no,Pre,3.0,1.0,III,30.0,721.0,False,999,0.000000
682,53.0,0.0,yes,Post,17.0,0.0,III,25.0,186.0,False,999,0.000000
683,51.0,0.0,no,Pre,5.0,43.0,III,25.0,769.0,True,1,0.001655
684,52.0,34.0,no,Post,3.0,15.0,II,23.0,727.0,True,0,0.001629


In [96]:
# Prepare df_train for sklearn DecisionTreeClassifier usage
# - encode categorical variables via one-hot
# - ensure numeric dtypes, no missing values
feature_cols = df_train.drop(["time", "event", "weights_ipcw", "survived"], axis=1).columns
X_df = df_train[feature_cols].copy()

# detect categorical/object columns and one-hot encode
cat_cols = X_df.select_dtypes(include=["category", "object"]).columns.tolist()
if cat_cols:
    X_df = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)

# fill missing values (numeric)
X_df = X_df.fillna(0)

# final arrays for sklearn
X = X_df.values.astype(float)
y = df_train["survived"].astype(int).values
sample_weights = df_train["weights_ipcw"].astype(float).values

# quick checks
print("X shape:", X.shape)
print("features:", list(X_df.columns)[:20])
print("y distribution:", np.bincount(y))
print("sample_weights sum:", sample_weights.sum())
X

X shape: (686, 9)
features: ['age', 'estrec', 'pnodes', 'progrec', 'tsize', 'horTh_yes', 'menostat_Post', 'tgrade_II', 'tgrade_III']
y distribution: [169 447   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0

array([[ 70.,  66.,   3., ...,   1.,   1.,   0.],
       [ 56.,  77.,   7., ...,   1.,   1.,   0.],
       [ 58., 271.,   9., ...,   1.,   1.,   0.],
       ...,
       [ 51.,   0.,   5., ...,   0.,   0.,   1.],
       [ 52.,  34.,   3., ...,   1.,   1.,   0.],
       [ 55.,  15.,   9., ...,   1.,   1.,   0.]], shape=(686, 9))

In [90]:
params_rf =         {   'n_estimators':len(df_train),                        
                        'max_depth':4,
                        'min_samples_split':5,
                        'max_features': 'log2',
                        'random_state':  42,
                        'weighted_bootstrapping': True, }

### Random Forest Modell ###
# Fit
clf = DecisionTreeBaggingClassifier(params_rf)
clf.fit(
    X=X,
    y=y,
    sample_weights=sample_weights,
)

# Evaluation auf den Trainingsdaten (verwende vorbereitete X)
_ , pred  = clf.predict_proba(X)
rf_test_mse = ipc_weighted_mse(
    y_true=y,
    y_pred=pred,
    sample_weight=sample_weights,
)

# Prediction für X_erwartung
#_ ,rf_pred = clf.predict_proba(data_generation_parameter['X_pred_point'].values)


ValueError: could not convert string to float: 'yes'